In [9]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load original dataset
df = pd.read_csv(
    r"C:\Users\denze\OneDrive\Desktop\Project\New folder\DataCo-Supply-Chain-Analytics\Data\Raw\DataCoSupplyChainDataset.csv",
    encoding="latin1",
    low_memory=False
)

# Work on a copy
df_clean = df.copy()

# Display original dataset dimensions
print("Original Dataset:", df_clean.shape)

# Display the first five rows
print(df_clean.head())


Original Dataset: (180519, 53)
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                         3                              4   
1  TRANSFER                         5                              4   
2      CASH                         4                              4   
3     DEBIT                         3                              4   
4   PAYMENT                         2                              4   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1        -249.089996          311.359985     Late delivery   
2        -247.779999          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk  Category Id   Category Name Customer City  ...  \
0                   0           73  Sporting Goods        Caguas  ...   
1                

In [10]:
columns_to_drop = [
    "Customer Email",
    "Customer Password",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Customer Zipcode",
    "Order Zipcode",
    "Product Description",
    "Product Image"
]

df_clean.drop(
    columns=columns_to_drop,
    inplace=True
)

print("Remaining Columns:", df_clean.shape[1])

Remaining Columns: 44


In [11]:
import re 
def clean_column_name(col):
    col = col.lower().strip()
    col = re.sub(r"[^a-z0-9]+","_", col)
    return col.strip("_")

    df_clean.columns = [
        clean_column_name(col)
        for col in df_clean.columns
    ]
    print(df_clean.columns.tolist())

In [12]:
missing = df_clean.isnull().sum()
print(missing[missing > 0])

Series([], dtype: int64)


In [13]:
print(
    "Remaining Missing Values:",
    df_clean.isnull().sum().sum()
)

Remaining Missing Values: 0


In [16]:
date_columns = [
    "order_date_dateorders",
    "shipping_date_dateorders"
]

for col in date_columns:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        format="%m/%d/%Y %H:%M",
        errors="coerce"
    )

print(df_clean[date_columns].dtypes)

print(
    "Invalid Dates:",
    df_clean[date_columns].isnull().sum()
)


order date (DateOrders)       datetime64[ns]
shipping date (DateOrders)    datetime64[ns]
dtype: object
Invalid or missing dates: order date (DateOrders)       0
shipping date (DateOrders)    0
dtype: int64


In [18]:
print(df_clean.columns.tolist())

['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Id', 'Customer Segment', 'Customer State', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Card Id', 'Product Category Id', 'Product Name', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']


In [19]:
print([
    col for col in df_clean.columns
    if "date" in col.lower()
])

['order date (DateOrders)', 'shipping date (DateOrders)']


In [20]:
import re
import pandas as pd

# Standardize all column names
df_clean.columns = [
    re.sub(
        r"[^a-z0-9]+",
        "_",
        col.lower().strip()
    ).strip("_")
    for col in df_clean.columns
]

# Verify the date columns
print([
    col for col in df_clean.columns
    if "date" in col
])

['order_date_dateorders', 'shipping_date_dateorders']


In [21]:
# Order date features
df_clean["order_year"] = df_clean["order_date_dateorders"].dt.year

df_clean["order_month"] = df_clean["order_date_dateorders"].dt.month

df_clean["order_quarter"] = df_clean["order_date_dateorders"].dt.quarter

df_clean["order_day"] = df_clean["order_date_dateorders"].dt.day

df_clean["order_weekday"] = df_clean["order_date_dateorders"].dt.day_name()

df_clean["order_month_name"] = df_clean["order_date_dateorders"].dt.month_name()

# Shipping date features
df_clean["shipping_year"] = df_clean["shipping_date_dateorders"].dt.year

df_clean["shipping_month"] = df_clean["shipping_date_dateorders"].dt.month

# Verify
display(
    df_clean[
        [
            "order_date_dateorders",
            "order_year",
            "order_month",
            "order_quarter",
            "order_weekday"
        ]
    ].head()
)

,order_date_dateorders,order_year,order_month,order_quarter,order_weekday
0,2018-01-31 22:56:00,2018,1,1,Wednesday
1,2018-01-13 12:27:00,2018,1,1,Saturday
2,2018-01-13 12:06:00,2018,1,1,Saturday
3,2018-01-13 11:45:00,2018,1,1,Saturday
4,2018-01-13 11:24:00,2018,1,1,Saturday


In [23]:
df_clean["delivery_delay_days"] = (
    df_clean["days_for_shipping_real"]
    - df_clean["days_for_shipment_scheduled"]
)

In [25]:
df_clean["delivery_performance"] = np.select(
    [
        df_clean["delivery_status"].eq(
            "Shipping canceled"
        ),
        df_clean["delivery_delay_days"] > 0,
        df_clean["delivery_delay_days"] < 0
    ],
    [
        "Canceled",
        "Late",
        "Early"
    ],
    default="On Time"
)


In [26]:
df_clean["on_time_delivery_flag"] = np.where(
    df_clean["delivery_performance"] == "Canceled",
    np.nan,
    np.where(
        df_clean["delivery_delay_days"] <= 0,
        1,
        0
    )
)

In [27]:
print(
    df_clean["delivery_performance"]
    .value_counts()
)

print(
    df_clean[
        [
            "delivery_status",
            "delivery_delay_days",
            "delivery_performance",
            "on_time_delivery_flag"
        ]
    ].head(10)
)

delivery_performance
Late        98977
Early       41592
On Time     32196
Canceled     7754
Name: count, dtype: int64
     delivery_status  delivery_delay_days delivery_performance  \
0   Advance shipping                   -1                Early   
1      Late delivery                    1                 Late   
2   Shipping on time                    0              On Time   
3   Advance shipping                   -1                Early   
4   Advance shipping                   -2                Early   
5  Shipping canceled                    2             Canceled   
6      Late delivery                    1                 Late   
7      Late delivery                    1                 Late   
8      Late delivery                    1                 Late   
9      Late delivery                    1                 Late   

   on_time_delivery_flag  
0                    1.0  
1                    0.0  
2                    1.0  
3                    1.0  
4                  

In [28]:
# Profit margin as a percentage
df_clean["profit_margin_pct"] = np.where(
    df_clean["order_item_total"] != 0,
    (
        df_clean["order_profit_per_order"]
        / df_clean["order_item_total"]
    ) * 100,
    np.nan
)

# Flag loss-making order-item records
df_clean["loss_flag"] = (
    df_clean["order_profit_per_order"] < 0
).astype(int)

In [29]:
df_clean["order_value_category"] = pd.cut(
    df_clean["order_item_total"],
    bins=[-np.inf, 100, 500, np.inf],
    labels=["Low", "Medium", "High"]
)

print(
    df_clean["order_value_category"]
    .value_counts()
)

order_value_category
Medium    137252
Low        42596
High         671
Name: count, dtype: int64


In [30]:
# Duplicate order-item IDs
print(
    "Duplicate Order Item IDs:",
    df_clean["order_item_id"].duplicated().sum()
)

# Missing dates
print(
    "Missing Order Dates:",
    df_clean["order_date_dateorders"].isna().sum()
)

# Negative quantities
print(
    "Negative Quantities:",
    (df_clean["order_item_quantity"] < 0).sum()
)

# Verify sales calculation
sales_difference = (
    df_clean["sales"]
    - df_clean["order_item_discount"]
    - df_clean["order_item_total"]
)

print(
    "Sales Calculation Mismatches:",
    (sales_difference.abs() > 0.02).sum()
)

# Check delivery status categories
print(
    df_clean["delivery_performance"].value_counts()
)

# Final dataset dimensions
print("Final Dataset Shape:", df_clean.shape)

Duplicate Order Item IDs: 0
Missing Order Dates: 0
Negative Quantities: 0
Sales Calculation Mismatches: 0
delivery_performance
Late        98977
Early       41592
On Time     32196
Canceled     7754
Name: count, dtype: int64
Final Dataset Shape: (180519, 58)


In [31]:
output_path = Path("../Data/Cleaned")
output_path.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(
    output_path / "DataCo_Cleaned.csv",
    index=False
)

print("Cleaned dataset successfully exported!")
print("Final Shape:", df_clean.shape)

Cleaned dataset successfully exported!
Final Shape: (180519, 58)


In [32]:
summary = {
    "Metric": [
        "Total Records",
        "Total Columns",
        "Unique Orders",
        "Unique Customers",
        "Unique Products",
        "Missing Values",
        "Duplicate Order Items"
    ],
    "Value": [
        len(df_clean),
        len(df_clean.columns),
        df_clean["order_id"].nunique(),
        df_clean["customer_id"].nunique(),
        df_clean["product_card_id"].nunique(),
        df_clean.isnull().sum().sum(),
        df_clean["order_item_id"].duplicated().sum()
    ]
}

summary_df = pd.DataFrame(summary)

display(summary_df)

summary_df.to_csv(
    output_path / "DataCo_Data_Quality_Report.csv",
    index=False
)

,Metric,Value
0,Total Records,180519
1,Total Columns,58
2,Unique Orders,65752
3,Unique Customers,20652
4,Unique Products,118
5,Missing Values,7754
6,Duplicate Order Items,0
